# Notebook 07: Voice Agent with Pipecat

**Time:** 30 minutes  
**Prerequisites:** Notebook 06 complete  
**Goal:** Build a voice agent pipeline: ASR -> LLM -> TTS, and explore Pipecat framework

This notebook will:
1. Build a manual voice pipeline (ASR -> LLM -> TTS) step by step
2. Introduce Pipecat -- the leading open-source voice agent framework
3. Design a conversation flow with multi-turn memory
4. Discuss production voice agent architecture

> **Why this matters:** Voice agents are the next frontier of AI applications. The original homework asked students to build a voice agent from scratch with FastAPI + Whisper + CosyVoice. In 2025, frameworks like Pipecat (5K+ GitHub stars) abstract away the complexity, letting you focus on the agent logic.

In [1]:
import os, sys, time, importlib
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'), override=True)

import src.llm_client, src.cost_tracker, src.utils, src.config
for mod in [src.llm_client, src.cost_tracker, src.utils, src.config]:
    importlib.reload(mod)

from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import format_response, append_to_reflection
import src.config as config

import src.audio_utils
importlib.reload(src.audio_utils)
from src.audio_utils import (
    transcribe_with_faster_whisper,
    synthesize_with_edge_tts,
)

client  = LLMClient(path=config.PATH)
tracker = CostTracker()

outputs_dir = os.path.join('..', 'outputs')
os.makedirs(outputs_dir, exist_ok=True)

print("Setup complete -- ready for Notebook 07")

✓ Ollama client initialized
  Available models: ['nomic-embed-text:latest', 'qwen3.5:latest', 'gemma4:e2b', 'llama3:latest', 'granite4:latest']
  Default model: granite4:latest
Setup complete -- ready for Notebook 07


---

## Part 1: Manual Voice Pipeline (ASR -> LLM -> TTS)

Before using a framework, let's build the pipeline manually to understand each component:

```
Audio Input -> [ASR: faster-whisper] -> Text -> [LLM: Claude/Ollama] -> Response -> [TTS: edge-tts] -> Audio Output
```

In [2]:
def voice_pipeline(
    audio_path: str,
    conversation_history: list,
    client: LLMClient,
    tracker: CostTracker,
    output_dir: str = "outputs",
):
    """
    Complete voice agent pipeline: Audio -> Text -> LLM -> Speech
    
    Args:
        audio_path: Path to input audio file
        conversation_history: List of {role, content} dicts for context
        client: LLMClient instance
        tracker: CostTracker instance
        output_dir: Directory for output audio
    
    Returns:
        Dict with 'user_text', 'bot_text', 'audio_path', 'timings'
    """
    timings = {}
    total_start = time.time()
    
    # Step 1: ASR (Speech-to-Text)
    print("[1/3] ASR: Transcribing audio...")
    asr_start = time.time()
    asr_result = transcribe_with_faster_whisper(
        audio_path, model_size="base", device="cpu", compute_type="int8"
    )
    user_text = asr_result['text']
    timings['asr'] = round(time.time() - asr_start, 2)
    print(f"  User said: {user_text}")
    
    # Step 2: LLM (Generate response with conversation history)
    print("\n[2/3] LLM: Generating response...")
    llm_start = time.time()
    
    conversation_history.append({"role": "user", "content": user_text})
    
    # Build context from history (last 5 turns)
    context = "\n".join(
        f"{turn['role']}: {turn['content']}"
        for turn in conversation_history[-10:]  # Last 5 turns (10 messages)
    )
    
    response = client.generate(
        prompt=f"Conversation so far:\n{context}\n\nRespond naturally and concisely as the assistant.",
        system="You are a helpful voice assistant. Keep responses under 3 sentences since they will be spoken aloud.",
        max_tokens=200,
        temperature=0.7
    )
    
    if "error" not in response:
        tracker.add_call(response)
        bot_text = response['content']
    else:
        bot_text = "I'm sorry, I encountered an error processing your request."
    
    timings['llm'] = round(time.time() - llm_start, 2)
    conversation_history.append({"role": "assistant", "content": bot_text})
    print(f"  Bot says: {bot_text}")
    
    # Step 3: TTS (Text-to-Speech)
    print("\n[3/3] TTS: Synthesizing speech...")
    tts_start = time.time()
    audio_output = os.path.join(output_dir, f'response_turn{len(conversation_history)//2}.mp3')
    tts_result = synthesize_with_edge_tts(
        text=bot_text,
        voice="en-US-AriaNeural",
        save_path=audio_output,
    )
    timings['tts'] = round(time.time() - tts_start, 2)
    timings['total'] = round(time.time() - total_start, 2)
    
    print(f"\n--- Pipeline Timing ---")
    print(f"  ASR:   {timings['asr']:.2f}s")
    print(f"  LLM:   {timings['llm']:.2f}s")
    print(f"  TTS:   {timings['tts']:.2f}s")
    print(f"  TOTAL: {timings['total']:.2f}s")
    
    return {
        'user_text': user_text,
        'bot_text': bot_text,
        'audio_path': audio_output,
        'timings': timings,
    }

print("Voice pipeline function defined.")

Voice pipeline function defined.


In [3]:
print("=" * 65)
print("Experiment 1: Run Voice Pipeline")
print("=" * 65)
print()

# Find test audio
test_audio = os.path.join(parent_dir, 'test_data', 'audio', 'sample-1.mp3')
if not os.path.exists(test_audio):
    test_audio = os.path.join(parent_dir, '..', 'MLE_in_Gen_AI-Course', 'Class3', 'test_data', 'audio', 'sample-1.mp3')

if os.path.exists(test_audio):
    conversation_history = []
    result = voice_pipeline(
        audio_path=test_audio,
        conversation_history=conversation_history,
        client=client,
        tracker=tracker,
        output_dir=outputs_dir,
    )
    
    # Play response
    from IPython.display import Audio, display
    if os.path.exists(result['audio_path']):
        display(Audio(result['audio_path']))
else:
    print("No test audio found. Place an audio file at: test_data/audio/sample-1.mp3")
    print("\nAlternatively, we'll simulate with text input below.")

Experiment 1: Run Voice Pipeline

[1/3] ASR: Transcribing audio...
FASTER-WHISPER ASR (model: base, cpu/int8)
Audio: /home/conardd/working_space/Homework3-Submission/test_data/audio/sample-1.mp3
  Language: en (prob: 0.687)
  Segments: 4
  Transcribed in 2.9s
  Text: We pay for that, they didn't, you know, that is some people's success on. Kren the Benzo, instead of Quotan and musical, there's not a story of what you're interested in just turning. That's the......
  User said: We pay for that, they didn't, you know, that is some people's success on. Kren the Benzo, instead of Quotan and musical, there's not a story of what you're interested in just turning. That's the...

[2/3] LLM: Generating response...
  Bot says: I'm sorry for the confusion, but it seems like your message got cut off or is quite jumbled. Can you please clarify or rephrase your statement? I'm here to assist you!

[3/3] TTS: Synthesizing speech...
EDGE TTS (voice: en-US-AriaNeural)
Text: I'm sorry for the confusion, 

---

## Part 2: Text-Based Voice Agent Simulation

Even without audio input, we can simulate the voice agent flow using text input and TTS output.

In [4]:
print("=" * 65)
print("Experiment 2: Simulated 3-Turn Conversation")
print("=" * 65)
print()

conversation = []
user_messages = [
    "What is pretraining data and why does it matter for LLMs?",
    "How do companies like Meta collect their training data?",
    "What's the biggest challenge in building a pretraining dataset?",
]

for i, user_msg in enumerate(user_messages):
    print(f"\n{'='*50}")
    print(f"Turn {i+1}")
    print(f"{'='*50}")
    print(f"User: {user_msg}")
    
    conversation.append({"role": "user", "content": user_msg})
    
    context = "\n".join(f"{t['role']}: {t['content']}" for t in conversation[-10:])
    
    start = time.time()
    response = client.generate(
        prompt=f"Conversation:\n{context}\n\nRespond concisely (2-3 sentences).",
        system="You are a voice assistant explaining ML concepts. Keep responses brief since they'll be spoken aloud.",
        max_tokens=150,
        temperature=0.7
    )
    elapsed = time.time() - start
    
    if "error" not in response:
        tracker.add_call(response)
        bot_text = response['content']
        conversation.append({"role": "assistant", "content": bot_text})
        print(f"Bot ({elapsed:.1f}s): {bot_text}")
        
        # Synthesize response
        tts_result = synthesize_with_edge_tts(
            text=bot_text,
            save_path=os.path.join(outputs_dir, f'sim_turn_{i+1}.mp3'),
        )
    else:
        print(f"Error: {response['error']}")

print(f"\nConversation history: {len(conversation)} messages")

Experiment 2: Simulated 3-Turn Conversation


Turn 1
User: What is pretraining data and why does it matter for LLMs?
Bot (5.2s): Pretraining data refers to the vast amounts of text used to train large language models (LLMs) before they're fine-tuned on specific tasks. It matters because a larger, more diverse dataset allows the model to learn a wider range of patterns and relationships in human language, which can significantly improve its performance on downstream tasks.
EDGE TTS (voice: en-US-AriaNeural)
Text: Pretraining data refers to the vast amounts of text used to train large language models (LLMs) befor...
  Synthesized in 0.6s
  Saved to: ../outputs/sim_turn_1.mp3

Turn 2
User: How do companies like Meta collect their training data?
Bot (4.9s): Companies like Meta often gather training data from publicly available sources such as books, websites, social media posts, and academic papers. They may also use web crawlers to collect additional text from the internet, ensuring a div

In [8]:
# TODO 1: Design your own voice agent conversation
#
# Create a 3-5 turn conversation with a specific persona.
# The agent should maintain context across turns.

print("=" * 65)
print("TODO 1: Custom Voice Agent")
print("=" * 65)
print()

my_persona = "['You are a professional coding tutor']"
my_questions = [
    "What is the best way to learn Python programming as a beginner?",
    "How long does it typically take to become proficient in Python?",
    "What are some common pitfalls to avoid when learning Python?",
]

my_conversation = []
for i, question in enumerate(my_questions):
    print(f"\nTurn {i+1}: {question}")
    my_conversation.append({"role": "user", "content": question})
    
    ctx = "\n".join(f"{t['role']}: {t['content']}" for t in my_conversation[-10:])
    
    resp = client.generate(
        prompt=f"Conversation:\n{ctx}\n\nRespond in 2-3 sentences.",
        system=my_persona,
        max_tokens=150,
        temperature=0.7
    )
    
    if "error" not in resp:
        tracker.add_call(resp)
        my_conversation.append({"role": "assistant", "content": resp['content']})
        print(f"  Bot: {resp['content']}")

todo1_reflection = """
[YOUR REFLECTION HERE]
Agent did maintain context across turns.
I ask the agent to be a tutor, and it did well.
I'd like to makethe persona more specific.
- Did the agent maintain context across turns?
- What persona did you choose and how well did it work?
- What would you change to make the conversation more natural?
"""

print()
print(todo1_reflection)

TODO 1: Custom Voice Agent


Turn 1: What is the best way to learn Python programming as a beginner?
  Bot: To learn Python programming as a beginner, start by familiarizing yourself with the basic syntax and concepts such as variables, data types, loops, conditionals, functions, and error handling. Practice writing simple programs and gradually work on more complex projects to reinforce your understanding of these fundamentals. Additionally, utilize online resources like tutorials, video courses, coding challenges, and interactive platforms to enhance your learning experience and gain practical hands-on experience with Python programming.

Turn 2: How long does it typically take to become proficient in Python?
  Bot: The time required to become proficient in Python varies depending on factors such as prior programming experience, dedication to learning, and the amount of practice one engages in. On average, a beginner can expect to spend several months to a year actively working with 

---

## Part 3: Pipecat Framework (2025)

**Pipecat** is the leading open-source framework for building real-time voice and multimodal agents. It orchestrates the ASR -> LLM -> TTS pipeline with:

- **Pipeline architecture**: Composable processing frames
- **Built-in transports**: WebRTC, WebSocket, telephone
- **Service integrations**: Whisper, Claude, OpenAI, ElevenLabs, Kokoro
- **Conversation management**: Turn detection, interruption handling

```
pip install pipecat-ai
```

### Pipecat vs Manual FastAPI (Original Homework Approach)

| Feature | Manual FastAPI | Pipecat |
|---------|---------------|--------|
| Setup complexity | High (write everything) | Low (composable pipeline) |
| Real-time streaming | Manual WebSocket | Built-in WebRTC |
| Turn detection | Manual VAD | Automatic |
| Interruption handling | Not supported | Built-in |
| Service switching | Code rewrite | Config change |

In [10]:
# TODO 2: Voice agent architecture analysis

print("=" * 65)
print("TODO 2: Voice Agent Architecture")
print("=" * 65)
print()

start = time.time()
response = client.generate(
    prompt="""I built a simple voice agent with this pipeline:
  Audio -> faster-whisper (ASR) -> Claude (LLM) -> edge-tts (TTS) -> Audio

Now I want to make it production-ready. Compare two approaches:

1. **Manual FastAPI**: Write a FastAPI server that handles audio uploads,
   runs ASR/LLM/TTS sequentially, returns audio response.

2. **Pipecat framework**: Use pipecat-ai to orchestrate the pipeline with
   built-in WebRTC transport, VAD, and turn detection.

For each approach, discuss:
- Development effort (hours/days)
- Latency characteristics
- Scalability
- When to choose one over the other

Also explain how Pipecat handles interruptions (user speaks while bot is still talking).""",
    system="You are a voice AI systems architect.",
    max_tokens=600,
    temperature=0.5
)
elapsed = time.time() - start

if "error" not in response:
    tracker.add_call(response)
    print(f"Response in {elapsed:.1f}s")
    print(format_response(response, verbose=True))
else:
    print(f"Error: {response['error']}")

todo2_reflection = """
[YOUR REFLECTION HERE]
I don't have experience with Pipecat, but as a framework it likely abstracts away a lot of the complexities of building a voice agent. I will try later.
It's mature technology anf easy to implement.
Stop the botwaiting for the user to finish speaking.
- Would you use Pipecat or build your own FastAPI voice server? Why?
- What's the most challenging part of building a real-time voice agent?
- How would you handle the case where the user interrupts the bot mid-sentence?
"""

print()
print(todo2_reflection)

TODO 2: Voice Agent Architecture

Response in 49.8s
Model: granite4:latest
Tokens: 187 in, 600 out
Stop reason: complete
Here's a comparison of the two approaches for making your voice agent production-ready:

### 1. Manual FastAPI Server

**Development Effort**: 
- Building a manual FastAPI server would require significant development effort.
- You would need to implement audio upload handling, ASR (using faster-whisper), LLM processing (using Claude), TTS (edge-tts), and then send the processed audio back.
- This could take anywhere from 10 to 30 days depending on your team's expertise and the complexity of the integration.

**Latency Characteristics**: 
- The latency would depend heavily on how you handle each step in the pipeline.
- ASR might introduce noticeable delay (around 1-2 seconds).
- LLM processing time could vary but generally takes a few seconds to process complex queries.
- TTS conversion is usually instantaneous.
- Overall, expect the end-to-end latency to be around 5-

---

## Summary & Reflection

In [11]:
_todo1 = todo1_reflection.strip() if 'todo1_reflection' in dir() else '[TODO 1 not completed yet]'
_todo2 = todo2_reflection.strip() if 'todo2_reflection' in dir() else '[TODO 2 not completed yet]'

full_reflection = f"""
### Part 1 - Custom Voice Agent

{_todo1}

---

### Part 2 - Production Architecture

{_todo2}
"""

reflection_file = append_to_reflection(
    notebook="07",
    section_title="Voice Agent with Pipecat",
    reflection_content=full_reflection,
    output_dir=os.path.join('..', 'outputs')
)

print(f"Reflection saved: {reflection_file}")
print()
tracker.report()

Reflection saved: ../outputs/homework_reflection.md

API COST REPORT
Total API calls:     18
Total input tokens:  2,821
Total output tokens: 2,597
Total cost:          $0.0474

Last 5 calls:
  1. [14:49:53] granite4:latest -- 46in/89out -- $0.0015
  2. [14:50:03] granite4:latest -- 150in/99out -- $0.0019
  3. [14:50:12] granite4:latest -- 263in/97out -- $0.0022
  4. [14:54:50] granite4:latest -- 187in/600out -- $0.0096
  5. [15:02:54] granite4:latest -- 187in/600out -- $0.0096


## Notebook 07 Complete!

**What you accomplished:**
- Built a complete ASR -> LLM -> TTS voice pipeline
- Simulated multi-turn voice conversations
- Explored Pipecat framework for production voice agents

**Key concepts:**
- Voice agents combine ASR, LLM, and TTS in a real-time pipeline
- Conversation history enables multi-turn context
- Pipecat abstracts pipeline complexity with composable frames

**Next:** Open **Notebook 08: Project Integration**